# Game Stats Dev Notebook

Build team stats and player stats for one game in notebook code, without saving to Mongo.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from typing import Any

import numpy as np
import pandas as pd
from pymongo import MongoClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "prod_pipeline").exists():
    ROOT = ROOT.parent
PROD_PIPELINE = ROOT / "prod_pipeline"
if str(PROD_PIPELINE) not in sys.path:
    sys.path.insert(0, str(PROD_PIPELINE))

from helper import load_app_config


In [ ]:
CONFIG_PATH = ROOT / "config" / "config.yaml"
events_config = load_app_config(str(CONFIG_PATH))

mongo_config = events_config.mongo
game_stats_config = events_config.game_stats

team_match_features = game_stats_config.team_match_features
player_match_features = game_stats_config.player_match_features
stat_columns = list(game_stats_config.match_stats)
pass_completion_base_columns = game_stats_config.pass_complettion_stats
perc_stats = game_stats_config.perc_stats
stat_rename = game_stats_config.match_stats_rename

events_config


In [ ]:
client = MongoClient(mongo_config.url)
db = client[mongo_config.db]

processed_events_collection = db[mongo_config.collections["collection_processed_events"]]
team_stats_collection = db[mongo_config.collections["collection_team_game_stats"]]
player_stats_collection = db[mongo_config.collections["collection_player_game_stats"]]


def processed_game_ids() -> list[int]:
    return sorted(
        int(game_id)
        for game_id in processed_events_collection.distinct("game_id", {"season": events_config.season.year})
        if game_id is not None
    )


def stats_game_ids() -> list[int]:
    team_ids = {
        int(game_id)
        for game_id in team_stats_collection.distinct("game_id", {"season": events_config.season.year})
        if game_id is not None
    }
    player_ids = {
        int(game_id)
        for game_id in player_stats_collection.distinct("game_id", {"season": events_config.season.year})
        if game_id is not None
    }
    return sorted(team_ids & player_ids)


def load_processed_game(game_id: int) -> pd.DataFrame:
    docs = list(
        processed_events_collection.find(
            {"season": events_config.season.year, "game_id": int(game_id)},
            {"_id": 0},
        )
    )
    return pd.DataFrame(docs)


all_processed_ids = processed_game_ids()
already_built_ids = set(stats_game_ids())
pending_ids = [game_id for game_id in all_processed_ids if game_id not in already_built_ids]

print(f"Processed games available: {len(all_processed_ids):,}")
print(f"Processed games without stats: {len(pending_ids):,}")
print("First pending ids:", pending_ids[:10])

GAME_ID = pending_ids[0] if pending_ids else (all_processed_ids[0] if all_processed_ids else None)
GAME_ID


In [ ]:
if GAME_ID is None:
    raise ValueError("No processed games were found for this season.")

processed_game = load_processed_game(GAME_ID)
print(f"game_id={GAME_ID} processed rows: {len(processed_game):,}")
processed_game.head()


In [ ]:
def safe_div(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)


def add_rate_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if {"shot_zone_6_yard_box", "shot_zone_penalty_area"}.issubset(df.columns):
        df["shot_zone_inside_box"] = df["shot_zone_6_yard_box"] + df["shot_zone_penalty_area"]
    if {"shot_right_foot", "shot_left_foot"}.issubset(df.columns):
        df["shot_foot"] = df["shot_right_foot"] + df["shot_left_foot"]

    for new_col, formula in perc_stats.items():
        numerator = formula["numerator"]
        denominator = formula["denominator"]
        if numerator not in df.columns or denominator not in df.columns:
            df[new_col] = np.nan
            continue
        df[new_col] = safe_div(df[numerator], df[denominator])
    return df


def prepare_game_events(events: pd.DataFrame) -> pd.DataFrame:
    if events.empty:
        return events

    events = events.copy()
    home_team_id = events["home_team_id"].iloc[0]
    away_team_id = events["away_team_id"].iloc[0]
    home_team_name = events["home_team_name"].iloc[0]
    away_team_name = events["away_team_name"].iloc[0]

    events["game_venue"] = events["team_id"].map({
        home_team_id: "home",
        away_team_id: "away",
    }).fillna("neutral")
    events["team_name"] = events["team_id"].map({
        home_team_id: home_team_name,
        away_team_id: away_team_name,
    }).fillna(events["team"])
    events["opponent_team_id"] = events["team_id"].map({
        home_team_id: away_team_id,
        away_team_id: home_team_id,
    })
    events["opponent_team_name"] = events["team_id"].map({
        home_team_id: away_team_name,
        away_team_id: home_team_name,
    })

    foul_mask = events["type"].eq("Foul")
    foul_committed = foul_mask & events["outcome_type"].eq("Unsuccessful")
    foul_suffered = foul_mask & events["outcome_type"].eq("Successful")
    events.loc[foul_mask, "foul_committed"] = foul_committed[foul_mask]
    events.loc[foul_mask, "foul_suffered"] = foul_suffered[foul_mask]
    if "foul_type" in events.columns:
        events.loc[foul_committed, "foul_type"] = "Foul Committed"
        events.loc[foul_suffered, "foul_type"] = "Foul Suffered"

    return events


def add_derived_event_stats(game_events: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    game_events = game_events.copy()
    derived_columns: list[str] = []
    if "pass_completed" not in game_events.columns:
        return game_events, derived_columns

    pass_completed = pd.to_numeric(game_events["pass_completed"], errors="coerce").fillna(0).astype(int)
    for col in pass_completion_base_columns:
        if col not in game_events.columns:
            continue
        completed_col = f"{col}_completed"
        base = pd.to_numeric(game_events[col], errors="coerce").fillna(0).astype(int)
        game_events[completed_col] = (base.eq(1) & pass_completed.eq(1)).astype(int)
        derived_columns.append(completed_col)
    return game_events, derived_columns


def append_goal_assist_rows(game_events: pd.DataFrame, stat_columns: list[str], *, for_player: bool) -> pd.DataFrame:
    if "pass_assist" not in stat_columns:
        return game_events

    game_events = game_events.copy()
    game_events["pass_assist"] = 0
    if not {"goal_assist", "related_player_id"}.issubset(game_events.columns):
        return game_events

    goal_assist = pd.to_numeric(game_events["goal_assist"], errors="coerce").fillna(0).eq(1)
    related_player_id = pd.to_numeric(game_events["related_player_id"], errors="coerce")
    assist_rows = game_events[goal_assist & related_player_id.notna()].copy()
    if assist_rows.empty:
        return game_events

    stat_cols = [col for col in stat_columns if col in assist_rows.columns]
    assist_rows[stat_cols] = 0
    assist_rows["pass_assist"] = 1
    assist_rows["stat_event_type"] = "assist"

    if for_player:
        player_lookup = (
            game_events[["game_id", "player_id", "player"]]
            .dropna(subset=["player_id"])
            .drop_duplicates(["game_id", "player_id"])
            .set_index(["game_id", "player_id"])["player"]
        )
        assist_rows["player_id"] = related_player_id.loc[assist_rows.index]
        lookup_keys = pd.MultiIndex.from_frame(assist_rows[["game_id", "player_id"]])
        assist_rows["player"] = player_lookup.reindex(lookup_keys).to_numpy()

        for col in ["starting_lineup", "minutes_played"]:
            if col not in game_events.columns:
                continue
            value_lookup = (
                game_events[["game_id", "player_id", col]]
                .dropna(subset=["player_id"])
                .drop_duplicates(["game_id", "player_id"])
                .set_index(["game_id", "player_id"])[col]
            )
            assist_rows[col] = value_lookup.reindex(lookup_keys).to_numpy()

    return pd.concat([game_events, assist_rows], ignore_index=True)


def interval_masks(game_events: pd.DataFrame) -> dict[str, pd.Series]:
    minute = pd.to_numeric(game_events["minute"], errors="coerce")
    period = game_events["period"].fillna("")
    first_half = period.eq("FirstHalf")
    second_half = period.eq("SecondHalf")
    return {
        "ft": pd.Series(True, index=game_events.index),
        "fh": first_half,
        "sh": second_half,
        "m_1_15": minute.lt(15),
        "m_16_30": minute.ge(15) & minute.lt(30),
        "m_31_45": first_half & minute.ge(30),
        "m_46_60": second_half & minute.lt(60),
        "m_61_75": second_half & minute.ge(60) & minute.lt(75),
        "m_76_90": second_half & minute.ge(75),
        "m_1_30": minute.lt(30),
        "m_31_60": (first_half & minute.ge(30)) | (second_half & minute.lt(60)),
        "m_61_90": second_half & minute.ge(60),
    }


def add_possession_stats(team_stats: pd.DataFrame) -> pd.DataFrame:
    team_stats = team_stats.copy()
    if "pass_attempt" not in team_stats.columns:
        team_stats["possession"] = np.nan
        return team_stats

    pass_total = team_stats.groupby("game_id")["pass_attempt"].transform("sum")
    team_stats["possession"] = safe_div(team_stats["pass_attempt"], pass_total)
    return team_stats


def stat_dict(row: pd.Series, stat_value_columns: list[str]) -> dict[str, Any]:
    out = {}
    for col in stat_value_columns:
        value = row.get(col)
        if isinstance(value, np.generic):
            value = value.item()
        try:
            if pd.isna(value):
                value = None
        except Exception:
            pass
        out[col] = value
    return out


def rename_output_stat_keys(stats: dict[str, Any]) -> dict[str, Any]:
    return {stat_rename.get(key, key): value for key, value in stats.items()}


def rename_interval_stat_keys(interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]]) -> dict[tuple[int, int], dict[str, dict[str, Any]]]:
    for interval_stats in interval_lookup.values():
        for interval_name, stats in interval_stats.items():
            interval_stats[interval_name] = rename_output_stat_keys(stats)
    return interval_lookup


def flatten_interval_stats(df: pd.DataFrame, stats_col: str = "stats") -> pd.DataFrame:
    rows = []
    base_cols = [col for col in df.columns if col != stats_col]
    for _, row in df.iterrows():
        stats = row.get(stats_col, {}) or {}
        for interval_name, values in stats.items():
            record = {col: row[col] for col in base_cols}
            record["interval"] = interval_name
            record.update(values or {})
            rows.append(record)
    return pd.DataFrame(rows)


In [ ]:
prepared_game = prepare_game_events(processed_game)
prepared_game[[
    "game_id", "period", "minute", "team_name", "opponent_team_name", "game_venue", "type"
]].head(12)


In [ ]:
def align_team_fouls(team_stats: pd.DataFrame) -> pd.DataFrame:
    required = {"game_id", "team_id", "opponent_team_id", "foul_committed", "foul_suffered"}
    if not required.issubset(team_stats.columns):
        return team_stats

    team_stats = team_stats.copy()
    committed_lookup = team_stats.set_index(["game_id", "team_id"])["foul_committed"]
    opponent_keys = pd.MultiIndex.from_frame(team_stats[["game_id", "opponent_team_id"]])
    team_stats["foul_suffered"] = committed_lookup.reindex(opponent_keys).fillna(0).to_numpy()
    return team_stats


def summarize_by_team(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    game_events, derived_columns = add_derived_event_stats(game_events)
    game_events = append_goal_assist_rows(game_events, stat_columns, for_player=False)
    effective_stat_columns = [col for col in [*stat_columns, *derived_columns] if col in game_events.columns]
    game_events[effective_stat_columns] = game_events[effective_stat_columns].apply(pd.to_numeric, errors="coerce").fillna(0)

    team_stats = (
        game_events.groupby(team_match_features, dropna=False)[effective_stat_columns]
        .sum()
        .reset_index()
    )
    team_stats = align_team_fouls(team_stats)
    team_stats = add_rate_columns(team_stats)
    return add_possession_stats(team_stats)


def add_score_stats(interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]], game_events: pd.DataFrame) -> None:
    team_pairs = game_events[["game_id", "team_id", "opponent_team_id"]].drop_duplicates()
    for _, team_row in team_pairs.iterrows():
        key = (int(team_row["game_id"]), int(team_row["team_id"]))
        opponent_key = (int(team_row["game_id"]), int(team_row["opponent_team_id"]))
        for interval_name, stats in interval_lookup.get(key, {}).items():
            opponent_stats = interval_lookup.get(opponent_key, {}).get(interval_name, {})
            goals = int(stats.get("shot_goal", 0) or 0) + int(opponent_stats.get("shot_own_goal", 0) or 0)
            possession = stats.pop("possession", None)
            stats.pop("goals_for", None)

            reordered_stats = {}
            if possession is not None:
                reordered_stats["possession"] = possession
            reordered_stats["goals"] = goals
            reordered_stats.update(stats)
            stats.clear()
            stats.update(reordered_stats)


def build_team_interval_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> dict[tuple[int, int], dict[str, dict[str, Any]]]:
    interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]] = {}
    for interval_name, mask in interval_masks(game_events).items():
        interval_events = game_events.loc[mask].copy()
        if interval_events.empty:
            continue

        interval_stats = summarize_by_team(interval_events, stat_columns)
        stat_value_columns = [col for col in interval_stats.columns if col not in team_match_features]
        for _, row in interval_stats.iterrows():
            key = (int(row["game_id"]), int(row["team_id"]))
            interval_lookup.setdefault(key, {})[interval_name] = stat_dict(row, stat_value_columns)

    add_score_stats(interval_lookup, game_events)
    return rename_interval_stat_keys(interval_lookup)


def build_game_team_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    feature_columns = [col for col in team_match_features if col in game_events.columns]
    if not feature_columns:
        raise ValueError("No valid team match feature columns found in config game_stats.team_match_features.")

    team_stats = game_events[feature_columns].drop_duplicates().reset_index(drop=True).copy()
    interval_lookup = build_team_interval_stats(game_events, stat_columns)
    team_stats["stats"] = team_stats.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["team_id"])), {}),
        axis=1,
    )
    team_stats["opp_stats"] = team_stats.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["opponent_team_id"])), {}),
        axis=1,
    )
    return team_stats[feature_columns + ["stats", "opp_stats"]]


In [ ]:
team_stats = build_game_team_stats(prepared_game, stat_columns)
print(f"Team rows: {len(team_stats):,}")
team_stats


In [ ]:
flatten_interval_stats(team_stats).head(30)


In [ ]:
def event_match_seconds(game_events: pd.DataFrame) -> pd.Series:
    minute = pd.to_numeric(game_events["minute"], errors="coerce").fillna(0)
    second = pd.to_numeric(game_events.get("second", 0), errors="coerce").fillna(0)
    return minute.mul(60).add(second)


def add_player_appearance_columns(game_events: pd.DataFrame) -> pd.DataFrame:
    game_events = game_events.copy()
    if game_events.empty or not {"game_id", "team_id", "player_id"}.issubset(game_events.columns):
        return game_events

    event_seconds = event_match_seconds(game_events)
    match_end_seconds = event_seconds.groupby(game_events["game_id"]).transform("max")
    player_key = ["game_id", "team_id", "player_id"]

    sub_on_seconds = event_seconds.where(game_events["type"].eq("SubstitutionOn")).groupby(
        [game_events[col] for col in player_key], dropna=False
    ).min()
    sub_off_seconds = event_seconds.where(game_events["type"].eq("SubstitutionOff")).groupby(
        [game_events[col] for col in player_key], dropna=False
    ).min()

    appearance = game_events[player_key].dropna().drop_duplicates().copy()
    appearance = appearance.merge(sub_on_seconds.rename("sub_on_seconds").reset_index(), on=player_key, how="left")
    appearance = appearance.merge(sub_off_seconds.rename("sub_off_seconds").reset_index(), on=player_key, how="left")
    appearance["starting_lineup"] = appearance["sub_on_seconds"].isna()

    match_end_by_game = game_events.assign(_match_end_seconds=match_end_seconds)[["game_id", "_match_end_seconds"]].drop_duplicates()
    appearance = appearance.merge(match_end_by_game, on="game_id", how="left")
    appearance["start_seconds"] = appearance["sub_on_seconds"].fillna(0)
    appearance["end_seconds"] = appearance["sub_off_seconds"].fillna(appearance["_match_end_seconds"])
    appearance["minutes_played"] = (
        (appearance["end_seconds"] - appearance["start_seconds"]).clip(lower=0).div(60).apply(np.ceil).astype(int)
    )

    appearance = appearance[player_key + ["starting_lineup", "minutes_played"]]
    game_events = game_events.merge(appearance, on=player_key, how="left")
    game_events["starting_lineup"] = game_events["starting_lineup"].fillna(False).astype(bool)
    game_events["minutes_played"] = pd.to_numeric(game_events["minutes_played"], errors="coerce").astype("Int64")
    return game_events


def append_goal_assist_player_identities(player_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    if "pass_assist" not in stat_columns:
        return player_events
    if not {"goal_assist", "related_player_id"}.issubset(player_events.columns):
        return player_events

    goal_assist = pd.to_numeric(player_events["goal_assist"], errors="coerce").fillna(0).eq(1)
    related_player_id = pd.to_numeric(player_events["related_player_id"], errors="coerce")
    identity_rows = player_events[goal_assist & related_player_id.notna()].copy()
    if identity_rows.empty:
        return player_events

    player_lookup = (
        player_events[["game_id", "player_id", "player"]]
        .dropna(subset=["player_id"])
        .drop_duplicates(["game_id", "player_id"])
        .set_index(["game_id", "player_id"])["player"]
    )
    identity_rows["player_id"] = related_player_id.loc[identity_rows.index]
    lookup_keys = pd.MultiIndex.from_frame(identity_rows[["game_id", "player_id"]])
    identity_rows["player"] = player_lookup.reindex(lookup_keys).to_numpy()
    identity_rows["goal_assist"] = 0

    for col in stat_columns:
        if col in identity_rows.columns:
            identity_rows[col] = 0

    for col in ["starting_lineup", "minutes_played"]:
        if col not in player_events.columns:
            continue
        value_lookup = (
            player_events[["game_id", "player_id", col]]
            .dropna(subset=["player_id"])
            .drop_duplicates(["game_id", "player_id"])
            .set_index(["game_id", "player_id"])[col]
        )
        identity_rows[col] = value_lookup.reindex(lookup_keys).to_numpy()

    return pd.concat([player_events, identity_rows], ignore_index=True)


def summarize_by_player(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    player_events = game_events[game_events["player_id"].notna()].copy()
    if player_events.empty:
        return pd.DataFrame()

    player_events, derived_columns = add_derived_event_stats(player_events)
    player_events = append_goal_assist_rows(player_events, stat_columns, for_player=True)
    effective_stat_columns = [col for col in [*stat_columns, *derived_columns] if col in player_events.columns]
    feature_columns = [col for col in player_match_features if col in player_events.columns]
    if not feature_columns:
        raise ValueError("No valid player match feature columns found in config game_stats.player_match_features.")
    if not effective_stat_columns:
        return player_events[feature_columns].drop_duplicates().reset_index(drop=True)

    player_events[effective_stat_columns] = player_events[effective_stat_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
    player_stats = (
        player_events.groupby(feature_columns, dropna=False)[effective_stat_columns]
        .sum()
        .reset_index()
    )
    return add_rate_columns(player_stats)


def build_player_interval_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> dict[tuple[int, int], dict[str, dict[str, Any]]]:
    interval_lookup: dict[tuple[int, int], dict[str, dict[str, Any]]] = {}
    for interval_name, mask in interval_masks(game_events).items():
        interval_events = game_events.loc[mask].copy()
        if interval_events.empty:
            continue

        interval_stats = summarize_by_player(interval_events, stat_columns)
        if interval_stats.empty:
            continue

        feature_columns = [col for col in player_match_features if col in interval_stats.columns]
        stat_value_columns = [col for col in interval_stats.columns if col not in feature_columns]
        for _, row in interval_stats.iterrows():
            key = (int(row["game_id"]), int(row["player_id"]))
            interval_lookup.setdefault(key, {})[interval_name] = stat_dict(row, stat_value_columns)
    return interval_lookup


def add_player_per_90_stats(stats: dict[str, dict[str, Any]], minutes_played: Any) -> dict[str, dict[str, Any]]:
    if "ft" not in stats:
        return stats

    minutes = pd.to_numeric(pd.Series([minutes_played]), errors="coerce").iloc[0]
    minutes = None if pd.isna(minutes) else float(minutes)
    per_90_stats: dict[str, Any] = {}
    for source_col, raw_value in stats["ft"].items():
        value = pd.to_numeric(pd.Series([raw_value]), errors="coerce").iloc[0]
        if pd.isna(value) or minutes is None or minutes == 0:
            per_90_stats[source_col] = None
        else:
            per_90_stats[source_col] = float(value) / minutes * 90

    stats["per_90"] = per_90_stats
    return stats


def build_game_player_stats(game_events: pd.DataFrame, stat_columns: list[str]) -> pd.DataFrame:
    player_events = add_player_appearance_columns(game_events)
    player_events = player_events[player_events["player_id"].notna()].copy()
    player_events = append_goal_assist_player_identities(player_events, stat_columns)

    feature_columns = [col for col in player_match_features if col in player_events.columns]
    if player_events.empty:
        return pd.DataFrame(columns=feature_columns + ["stats"])
    if not feature_columns:
        raise ValueError("No valid player match feature columns found in config game_stats.player_match_features.")

    players = player_events[feature_columns].drop_duplicates().reset_index(drop=True).copy()
    interval_lookup = build_player_interval_stats(player_events, stat_columns)
    players["stats"] = players.apply(
        lambda row: interval_lookup.get((int(row["game_id"]), int(row["player_id"])), {}),
        axis=1,
    )
    players["stats"] = players.apply(
        lambda row: add_player_per_90_stats(row["stats"], row.get("minutes_played")),
        axis=1,
    )
    players["stats"] = players["stats"].apply(
        lambda stats: rename_interval_stat_keys({(0, 0): stats})[(0, 0)]
    )
    return players[feature_columns + ["stats"]]


In [ ]:
player_stats = build_game_player_stats(prepared_game, stat_columns)
print(f"Player rows: {len(player_stats):,}")
player_stats.head()


In [ ]:
flatten_interval_stats(player_stats).head(40)


In [ ]:
client.close()
